# 00 · SHAP attribution across FTTL versions — real data

**What this notebook does NOT do: it never opens a model pickle.** A pickle only unpickles inside
the env it was serialised in, and the three FTTL versions have mutually incompatible library stacks
(`src/docs/ENV_MANAGEMENT.md`). So the split is:

| step | where it runs | what it produces |
|---|---|---|
| compute per-row φ | `src/envs/v<k>/.venv/bin/python src/scoring/attribute.py` — **once per version, in that version's own env** | `src/data/real/detection/shap/<v>_attributions_<split>.parquet` (+ `_<suffix>` for a second backend on the same split) + `..._meta.json` |
| **this notebook** | the shared analysis `.venv` (kernel `ml-sfp-detection`) | tables + figures, for THREE configurations at once |

Run the compute step first — one command block per configuration this notebook reads (§0 below
lists exactly which):

```bash
python src/scoring/attribute_all.py --split train --rows 5000 --background 500
python src/scoring/attribute_all.py --split v1=val2 v2=test v3=oot --rows 5000 --background 500
python src/scoring/attribute_all.py --split v1=val2 v2=test v3=oot --backend native --out-suffix _native
```

**Which split, and why it is named.** The φ files exist only per split. Split names are each
version's own — v2's holdout is `test`, v3's is `oot`, v1 has `val1`/`val2` — and they are never
unified. It is load-bearing: concentration measured on `train` describes the fitted function on
data it saw, on a holdout it describes generalisation, and comparing one against the other is a
confound, not a finding — which is exactly the comparison §0 below sets up on purpose, tagged as
such rather than left implicit.

φ values are just numbers once they are on disk, so everything below is version-agnostic and the
dependency problem disappears. Nothing here hard-codes a path, a repo name or a column name —
they all come from `src/config.py` via `loaders.load()`.

**Read every number below at the model's own raw column names.** Feature names are the model's own raw column names. v1's 55
`make_*` columns and v2's 41 are *not* collapsed into one `make` — doing so would change the
concentration statistic this chapter rests on. Cross-version correspondence comes **only** from the
hand-confirmed mapping (`features/check_overlap.py` → `features/feature_overlap.json`), never from
name equality.


In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config           # noqa: E402  — real paths/columns live here and nowhere else
import schema           # noqa: E402
import figstyle         # noqa: E402
from loaders import load                       # noqa: E402
from estimator import concentration as conc    # noqa: E402

figstyle.apply()

SOURCE   = "real"
VERSIONS = list(config.VERSION_LABELS)   # trim to e.g. ["v2", "v3"] to compare a single pair

TOPN = 15    # features shown per version in the importance panels
TOPK = 5     # k for top-k share

print(f"repo root : {ROOT}")
print(f"versions  : {', '.join(VERSIONS)}  (source={SOURCE})")


## 0 · The three configurations this notebook compares

Every section below runs three times — once per row — instead of once for a single hard-coded
split:

| run | split | backend / perturbation | question it answers |
|---|---|---|---|
| `interventional_train` | each version's `"train"` split — the one split name spelled identically across all three (`config.SPLITS`) | `shap`, interventional, fixed background | concentration on data the fit saw |
| `interventional_oot` | each version's own out-of-time holdout (`config.OOT_SPLIT`: v1 `val2`, v2 `test`, v3 `oot`) | `shap`, interventional | concentration on data the fit did NOT see — this notebook's previous (and only) default |
| `path_dependent` | the SAME split as `interventional_oot` | `native` — each estimator's own TreeSHAP, tree-path-dependent | isolates the BACKEND as the only variable: does the reference/perturbation choice itself move the concentration numbers, split held fixed? |

`path_dependent` deliberately reuses `interventional_oot`'s split rather than a split of its own —
comparing it to `interventional_oot` is only a clean backend-only comparison if the split does not
also change. (A path-dependent run on `train` is one line away — call `analyze()` again with
`TRAIN_SPLITS` and `out_suffix="_native_train"`.)

**Why `path_dependent` needs `--out-suffix`.** `config.path("attributions", ...)` is keyed only by
`(version, split)`, not by backend — so attributing OOT under `native` after already attributing it
under `shap` would silently overwrite the first parquet at the identical path. `attribute_all.py`
gained `--out-suffix` for exactly this case. The `path_dependent` run below reads that suffixed file
by an explicit path, bypassing `loaders.load()`'s normal (unsuffixed) resolution — for
`.attributions`/`.attribution_meta` only. `.frame`/`.decisions`/`.tau` (scores, targets, the decision
rule) do not depend on the SHAP backend and still resolve the normal way through the same
`VersionData` object.


In [ ]:
TRAIN_SPLITS = {v: "train" for v in VERSIONS}   # spelled identically across config.SPLITS
OOT_SPLITS   = dict(config.OOT_SPLIT)           # v1 val2 · v2 test · v3 oot — each version's own name

for v in VERSIONS:
    assert TRAIN_SPLITS[v] in config.SPLITS[v], (v, TRAIN_SPLITS[v], config.SPLITS[v])
    assert OOT_SPLITS[v] in config.SPLITS[v], (v, OOT_SPLITS[v], config.SPLITS[v])

RUNS = {
    "interventional_train": {"splits": TRAIN_SPLITS, "out_suffix": ""},
    "interventional_oot":   {"splits": OOT_SPLITS,   "out_suffix": ""},
    "path_dependent":       {"splits": OOT_SPLITS,   "out_suffix": "_native"},
}


def _attributions_path(version: str, split: str, out_suffix: str) -> Path:
    """Mirror of attribute_all.py's own --out-suffix naming — the one other place that
    convention needs to exist, since a notebook cannot import a CLI driver's local variable."""
    base = config.path("attributions", version, SOURCE, split=split)
    return base.with_name(base.stem + out_suffix + base.suffix) if out_suffix else base


def _read_suffixed(version: str, split: str, out_suffix: str) -> tuple[pd.DataFrame, dict]:
    """Read an attributions parquet + sidecar meta by an EXPLICIT path — needed only for a
    suffixed (non-default-backend) file, which VersionData has no parameter for."""
    p = _attributions_path(version, split, out_suffix)
    meta_p = p.with_name(p.stem + "_meta.json")
    if not p.exists():
        raise FileNotFoundError(
            f"[{version}/{split}{out_suffix}] {p} not found. Produce it with:\n"
            f"    python src/scoring/attribute_all.py --split {version}={split} "
            f"--backend native --out-suffix {out_suffix!r} --versions {version}\n"
        )
    return pd.read_parquet(p), json.loads(meta_p.read_text(encoding="utf-8"))


## 1 · `analyze()` — the original §1/§2, made callable per run

Everything the notebook did in one pass before is now a function of `(run_label, splits,
out_suffix)`. Figures/prints are tagged with `run_label` so three executions in one kernel don't
overwrite each other's PNGs or read as one blurred comparison.

For a non-empty `out_suffix`, `.attributions`/`.attribution_meta` are read from the suffixed path
and assigned onto the `VersionData` instance BEFORE they are ever touched —
`functools.cached_property` treats a direct assignment exactly like a cache hit (it is a
non-data descriptor: instance-`__dict__` wins), so the normal `attribute.py`-produced file is never
read for that run, and `.frame`/`.decisions` still resolve the ordinary way through the same object.


In [ ]:
def analyze(run_label: str, splits: dict[str, str], out_suffix: str = "") -> dict:
    avail, missing = {}, {}
    for v in VERSIONS:
        d = load(v, SOURCE, split=splits[v])
        try:
            if out_suffix:
                phi, meta = _read_suffixed(v, splits[v], out_suffix)
                d.attributions = phi              # pre-seeds the cached_property — see note above
                d.attribution_meta = meta
            else:
                d.attributions                     # forces the default read; raises if absent
            avail[v] = d
        except FileNotFoundError as exc:
            missing[v] = str(exc).replace("\n", " ")

    print(f"\n=== {run_label}  ({', '.join(f'{v}/{splits[v]}' for v in VERSIONS)}"
          f"{out_suffix} ===")
    for v, why in missing.items():
        print(f"[{v}] NOT attributed yet — {why}\n")
    if not avail:
        print(f"[{run_label}] no version has attributions on disk — skipping.")
        return {"avail": {}, "metas": {}, "MABS": {}}

    metas = {v: d.attribution_meta for v, d in avail.items()}
    conc.require_comparable(metas)         # raises on mixed backends WITHIN this run

    FIELDS = ("split", "backend", "perturbation", "model_output", "estimator", "feature_order",
              "n_rows", "n_features", "background_n", "base_value")
    display(pd.DataFrame({v: {k: m.get(k) for k in FIELDS} for v, m in metas.items()}).T)

    MABS = {v: d.mean_abs_shap for v, d in avail.items()}     # mean|φ| per feature, descending

    n = len(MABS)
    fig, axes = plt.subplots(1, n, figsize=(max(figstyle.FIG_1[0], 4.0 * n), 0.30 * TOPN + 1.6))
    for ax, (v, m), colour in zip(np.atleast_1d(axes), MABS.items(), figstyle.SERIES):
        top = m.head(TOPN)[::-1]
        ax.barh(np.arange(len(top)), top.values, color=colour)
        ax.set_yticks(np.arange(len(top)))
        ax.set_yticklabels(top.index, fontsize=7)
        ax.set_title(f"{v} — top {TOPN} of {len(m)}")
        ax.set_xlabel("mean |φ|  (log-odds)")
    fig.suptitle(f"Attribution mass by feature — {run_label}")
    fig.tight_layout()
    figstyle.save(fig, f"00_real_shap_global_importance_{run_label}")
    plt.show()

    return {"avail": avail, "metas": metas, "MABS": MABS}


RESULTS = {name: analyze(name, cfg["splits"], cfg["out_suffix"]) for name, cfg in RUNS.items()}

assert any(res["avail"] for res in RESULTS.values()), (
    "no run produced any attributions. Run the three attribute_all.py command blocks in §0, "
    "then re-run this notebook.")


## 2 · Feature mapping — computed once, not per run

A version's trained feature *columns* don't change with which split or backend attributed them —
only the φ magnitudes do. So the hand-confirmed mapping (originally §3) is resolved once, against
the union of whichever versions any run managed to load, and reused by every run's concentration
tables below.


In [ ]:
ALL_MABS = {}
for res in RESULTS.values():
    ALL_MABS.update(res["MABS"])   # a version's raw column set is the SAME regardless of which
                                   # run attributed it — later runs just confirm the same index

names = {v: set(m.index) for v, m in ALL_MABS.items()}

# ── (a) raw string-equality intersection — the encoding-divergence EXHIBIT, not the basis
RAW_INTER = set.intersection(*names.values()) if len(names) > 1 else set(next(iter(names.values())))
print(f"raw string intersection across {list(names)}: {len(RAW_INTER)} features")
print("  (expected to be small — same concepts, different encodings. Comparison uses the")
print("   hand-confirmed mapping below, never name equality.)\n")

# ── (b) the hand-confirmed mapping: {"<idx>": {"v1": name, "v2": name, "v3": name}, ...}
overlap_json = ROOT / "features" / "feature_overlap.json"
MAPPING = {}     # idx -> {version: that version's own raw column name}
SHARED = {}      # version -> the set of ITS OWN names for rows mapped in every loaded version
if overlap_json.exists():
    MAPPING = {int(k): row for k, row in json.loads(overlap_json.read_text()).items()}
    have = list(ALL_MABS)
    full_rows = {k: row for k, row in MAPPING.items()
                 if all(v in row and row[v] in names[v] for v in have)}
    dropped = [k for k, row in MAPPING.items()
               if all(v in row for v in have) and k not in full_rows]
    if dropped:
        print(f"⚠ {len(dropped)} mapped rows name features ABSENT from the attributions "
              f"{dropped[:8]}{'…' if len(dropped) > 8 else ''} — stale mapping or wrong matrix; "
              f"resolve before reporting.")
    SHARED = {v: {row[v] for row in full_rows.values()} for v in have}
    print(f"hand-confirmed mapping: {len(MAPPING)} entries; {len(full_rows)} usable across "
          f"{have} and present in the attributions")
    for v in have:
        held = float(ALL_MABS[v][ALL_MABS[v].index.isin(SHARED[v])].sum() / ALL_MABS[v].sum()) if SHARED[v] else 0.0
        print(f"  {v}: {len(names[v]):>4} raw features · {len(SHARED[v]):>3} mapped · "
              f"mapped set holds {held:.1%} of its attribution mass")
    if len(have) > 1:
        pairs = pd.DataFrame(
            [[sum(1 for row in MAPPING.values() if a in row and b in row) for b in have]
             for a in have], index=have, columns=have)
        print("\nmapped features per version pair (diagonal = that version's mapped total):")
        display(pairs)
else:
    print(f"{overlap_json} not built yet — run features/check_overlap.py (needs the Excel; "
          f"company laptop).\nFalling back to the RAW intersection: every restricted number "
          f"below is an UNDERCOUNT — do not report it.")
    SHARED = {v: set(RAW_INTER) for v in ALL_MABS}

# the restricted comparisons are sized by construction: every SHARED[v] has the same length
N_SHARED = min((len(s) for s in SHARED.values()), default=0)


## 3 · Concentration, per run

Two tables per run — all features, and restricted to the mapped set (§2) — exactly what the
original notebook produced once, now looped over the three configurations.

Direction of the hypothesis, unchanged: if the loop is concentrating the model's reasoning onto the
fast-track drivers, later versions should show **lower** D1/D2 and **higher** Simpson / Gini /
top-k share. Compare row-by-row **within a version**: `interventional_train` vs
`interventional_oot` is the temporal question (does the fitted function's concentration generalise,
or is train-only overfitting driving it); `interventional_oot` vs `path_dependent` is the backend
question (is the concentration finding an artefact of the reference distribution). §5 turns this
into one table.


In [ ]:
PROFILES = {}
for name, res in RESULTS.items():
    if not res["MABS"]:
        continue
    full = conc.profile_table(res["MABS"], k=TOPK)
    print(f"\n[{name}] all features (D0 differs by construction — compare evenness, not D1/D2)")
    display(full.round(4))

    restricted = None
    if N_SHARED >= 5 and len(res["MABS"]) > 1:
        shared = {v: m[m.index.isin(SHARED[v])] for v, m in res["MABS"].items()}
        restricted = conc.profile_table(shared, k=TOPK)
        print(f"[{name}] restricted to the {N_SHARED} mapped features — D1/D2 directly comparable.")
        display(restricted.round(4))
    else:
        print(f"[{name}] mapped coverage is {N_SHARED} features — too thin for a restricted "
              f"comparison; per-version profiles only.")
    PROFILES[name] = {"full": full, "restricted": restricted}


### 3b · The configuration that produced those numbers — also computed once

`problem.md` §1.4c: **no adjacent version pair shares a configuration.** This is a property of the
fitted estimator, not of which split or backend attributed it, so — unlike §3 — there is exactly
one params table, not three.


In [ ]:
ALL_METAS = {}
for res in RESULTS.values():
    ALL_METAS.update(res["metas"])   # same fitted model regardless of split/backend, so the
                                     # params agree across runs; last write is as good as any

KNOBS = ("n_estimators", "max_depth", "max_leaves", "min_child_weight", "learning_rate", "eta",
         "reg_alpha", "reg_lambda", "gamma", "subsample", "colsample_bytree",
         "scale_pos_weight", "eval_metric", "objective")

params = {v: m.get("estimator_params") or {} for v, m in ALL_METAS.items()}
if any(params.values()):
    cfg = pd.DataFrame({v: {k: p.get(k, "—") for k in KNOBS} for v, p in params.items()})
    display(cfg)
    if len(params) > 1:
        differs = [k for k in KNOBS if len({str(p.get(k)) for p in params.values()}) > 1]
        if differs:
            print(f"knobs that DIFFER: {differs}\n"
                  "The concentration difference above is confounded with these — report it as "
                  "*consistent with* the loop, not as identifying it, unless a matched-"
                  "hyperparameter refit or a sensitivity sweep bounds them (problem.md §1.4c i–iii).")
        else:
            print("these versions are configuration-matched on the knobs above — the "
                  "regularisation confound of problem.md §1.4c does not apply to this pair.")
else:
    print("no estimator params in the meta — re-run attribute_all.py to capture them "
          "(they can only be read where the pickle opens).")


## 4 · Diversity profile — the three runs overlaid, one panel per version

The original notebook drew one curve per VERSION for a single configuration. This draws one PANEL
per version with the three RUNS as separate curves inside it — the shape a "does the story change"
question needs. Curves diverging at small q (the tail) vs large q (the dominant features) say
different things; the scalar D1/D2 numbers in §3 are a summary of exactly this picture.


In [ ]:
vset = sorted(set().union(*(res["MABS"].keys() for res in RESULTS.values())))
fig, axes = plt.subplots(1, len(vset), figsize=(4.2 * len(vset), 3.6), squeeze=False)
axes = axes[0]
for ax, v in zip(axes, vset):
    for (name, res), colour in zip(RESULTS.items(), figstyle.SERIES):
        if v not in res["MABS"]:
            continue
        m = res["MABS"][v]
        basis = m[m.index.isin(SHARED.get(v, set()))] if N_SHARED >= 5 else m
        curve = conc.hill_curve(basis)
        ax.plot(curve.index, curve.values, color=colour, label=name)
    ax.set_title(v)
    ax.set_xlabel("order  q")
    ax.set_yscale("log")
axes[0].set_ylabel("effective number of features  Dq")
axes[-1].legend(title="run", fontsize=7)
fig.suptitle("Attribution diversity profile — train vs OOT vs path-dependent backend"
             + (f" ({N_SHARED} mapped features)" if N_SHARED >= 5 else " (each version's own features)"))
fig.tight_layout()
figstyle.save(fig, "00_real_shap_hill_profile_by_run")
plt.show()


## 5 · Cross-run comparison — one table, the direct answer to "how does it differ"

`evenness_D2_D0` (dominance-weighted, unit-free) is the single number the SFP direction hypothesis
rests on — lower under the loop hypothesis. Reading DOWN a version's three rows: `train → oot`
should look similar if the loop is a property of the fitted function and not of the specific split;
a large jump questions that. Reading `interventional_oot → path_dependent`: a large jump means the
*backend* choice is doing real work on these numbers, and every concentration figure in this
notebook needs that caveat attached (the same point `00_SHAP.ipynb` §4 makes for the single-version
case).


In [ ]:
COLS = ("evenness_D1_D0", "evenness_D2_D0", "simpson_S", "gini", f"top{TOPK}_share")
rows = {}
for name, prof in PROFILES.items():
    table = prof["restricted"] if prof["restricted"] is not None else prof["full"]
    for v in table.index:
        rows[(v, name)] = {c: table.loc[v, c] for c in COLS if c in table.columns}
comparison = pd.DataFrame(rows).T
comparison.index.names = ["version", "run"]
display(comparison.round(4).sort_index())


## 6 · Concentration either side of the fast-track cutoff

The SFP mechanism only operates on claims that were **scrapped** — those are the rows whose label
was forced. So the sharper version of the question is whether concentration differs between the
scrapped and the garage-assessed side of that version's own rule (v1 segmented on mobility, v2
piecewise in time, v3 global — `threshold.apply` dispatches; nothing here assumes a single τ).

This is descriptive, not causal: the two sides differ in case-mix as well as in treatment. It is
the input to the estimator layer, not a result on its own.

Shown below for `interventional_oot` only, matching the original notebook's default —
`decision_side_profile()` takes any run label, so the other two are one call away.


In [ ]:
def decision_side_profile(run_label: str):
    res = RESULTS.get(run_label, {})
    if not res.get("avail"):
        print(f"[{run_label}] nothing loaded — skipping.")
        return None

    rows = {}
    for v, d in res["avail"].items():
        # PER-VERSION guard, not one try around the whole loop. `d.decisions` reproduces that
        # version's own rule and needs artefacts that do not all exist: v1's rule is segmented on
        # mobility and `VersionData.decisions` reads it from the production LOG — which v1 does
        # not have and never will (destroyed; `paths.log_source = None`). A single try/except
        # around the loop let v1's FileNotFoundError discard v2's and v3's rows too, turning a
        # known-missing artefact into an empty table.
        try:
            frame = d.frame[[schema.CLAIM_ID]].copy()
            frame["decision"] = d.decisions
        except (FileNotFoundError, KeyError, ValueError) as exc:
            print(f"[{run_label}/{v}] decisions cannot be reproduced — {str(exc).splitlines()[0]}")
            continue

        att = d.attributions.merge(frame, on=schema.CLAIM_ID, how="inner")
        if att.empty:
            print(f"[{run_label}/{v}] no attributed claim appears in the scored frame — skipping")
            continue
        for side, sub in att.groupby("decision"):
            label = "scrapped" if side == 1 else "garage"
            if len(sub) < 50:
                print(f"[{run_label}/{v}] {label}: only {len(sub)} rows — too thin, skipping")
                continue
            m = conc.mean_abs(sub.drop(columns=["decision"]), id_col=schema.CLAIM_ID)
            if N_SHARED >= 5:
                m = m[m.index.isin(SHARED.get(v, set()))]    # this version's OWN mapped names
            if m.empty or float(m.sum()) <= 0:
                print(f"[{run_label}/{v}] {label}: no attribution mass on the mapped features — skipping")
                continue
            rows[(v, label)] = {**conc.profile(m, k=TOPK), "n": len(sub)}

    if not rows:
        print(f"[{run_label}] no version produced a usable decision-side profile.")
        return None
    by_side = pd.DataFrame(rows).T
    display(by_side.round(4))
    return by_side


by_side_oot = decision_side_profile("interventional_oot")
# by_side_train    = decision_side_profile("interventional_train")   # uncomment to compare
# by_side_pathdep  = decision_side_profile("path_dependent")


## Notes, and what would invalidate this

- **Three configurations, one figure set.** `interventional_train` / `interventional_oot` /
  `path_dependent` (§0) are not alternatives to pick between — the point of this notebook version is
  that all three run together and §5 puts them in one table. A large move between
  `interventional_train` and `interventional_oot` says the concentration finding is split-sensitive;
  a large move between `interventional_oot` and `path_dependent` says it is backend-sensitive. Either
  weakens the finding; neither is fatal on its own, but both must be reported alongside it.
- **Backend.** If a run reports `perturbation = tree_path_dependent` (always true for
  `path_dependent`), the reference is each tree's own cover statistics, so part of any
  cross-version difference under that run is a difference in training distributions, not only in
  the fitted functions. `interventional_*` uses a fixed shared background instead — prefer it, and
  keep `path_dependent` as the robustness check it is here, not the headline number.
- **Rows.** No claim is common to all three versions — v2 runs 2018-01→2020-09 and v3 runs
  2023-06→2026-05, ~2¾ years apart. So `attribute_all.py` samples **per version by default**, and
  every difference here is confounded with case-mix. That caveat is standing, not conditional.
  `--shared-claims` exists only for a pair that overlaps in time (v1/v2) and refuses an empty
  intersection rather than silently sampling.
- **Windows.** v2 and v3 were trained years apart (README), so concentration differences are
  *descriptive* of the fitted functions, not evidence of the loop on their own — the
  identification argument is the DiD in `04_02`, and its parallel-trends assumption is the thing
  to defend.
- **`config.path()` is still keyed by `(version, split)` only.** `--out-suffix` is a notebook-local
  workaround for comparing backends on one split, not a general second axis — it exists on
  `attribute_all.py`'s CLI but nowhere in `config.py`/`loaders.py`, so nothing downstream of this
  notebook resolves a suffixed file automatically. If a suffixed run becomes a permanent fixture
  rather than a one-off robustness check, promote it to a real `config.SPLIT_KINDS`-style axis
  instead of growing more of these local readers.
- **The mapping is the only bridge.** If §2 fell back to the raw string intersection, nothing
  restricted may be reported. And if a future edit ever ranks `make` rather than `make_FORD`, the
  number has changed meaning — collapse never happens here; correspondence comes only from
  `features/feature_overlap.json` (hand-confirmed; typo-checked against the registries by
  `features/check_overlap.py`).
